# 半导体 PPI 与 SOXL 相对 SOXX 的一个月信息

固定问题、来源、日期版本、价格、预算和信息门见 `plan.json`。每个 ALFRED 日期只在下一 Qlib 交易日生成一个事件；完整市场历史为复用诊断。


In [ ]:
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root: raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/semiconductor_ppi_leverage_information_20261022"
plan_path=card/"plan.json"
plan=json.loads(plan_path.read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized"]
assert not (card/"run_state.json").exists() and not (card/"summary.json").exists(),"fixed attempt already started"
source=plan["source"]
source_card=root/source["card"]
for name,key in [("summary.json","summary_sha256"),("source_quality.json","source_quality_sha256"),
                 ("verification_manifest.json","verification_manifest_sha256"),
                 ("nas_readback.json","NAS_readback_sha256")]:
    assert sha(source_card/name)==source[key],name
for relative,digest in plan["provider"]["files_sha256"].items():
    assert sha(root/relative)==digest,relative
import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.core.recorder_sources import load_verified_recorder_objects
from src.workflow.inventory_sales_soxx_h63_information import stationary_bootstrap_lower
qlib.init(provider_uri=str(root/plan["provider"]["uri"]),region="us",kernels=1)
objects,source_audit=load_verified_recorder_objects(source_card,
    expected_recorder_id=source["recorder_id"],artifacts=[source["artifact"]],
    expected_artifact_sha256={source["artifact"]:source["artifact_sha256"]},
    expected_manifest_sha256=source["verification_manifest_sha256"])
release=objects[source["artifact"]].copy()
assert len(release)==93 and release.vintage_date.is_unique
assert release.next_qlib_session_0946_NY.is_unique
assert (release.next_qlib_session_0946_NY>release.vintage_date).all()
cal=pd.DatetimeIndex(D.calendar(start_time=plan["provider"]["calendar_bounds"][0],
    end_time=plan["provider"]["calendar_bounds"][1],freq="day")).normalize()
assert cal.is_unique and cal.is_monotonic_increasing
assert release.next_qlib_session_0946_NY.isin(cal).all()
assert release.yoy_pct_same_vintage.notna().sum()==92
assert not release.latest_observation_month.eq(pd.Timestamp("2025-10-01")).any()
events=release.set_index("next_qlib_session_0946_NY").sort_index()
train_event=events.loc[plan["event_windows"]["train_signal"][0]:plan["event_windows"]["train_signal"][1]].copy()
held_event=events.loc[plan["event_windows"]["held_reused_diagnostic"][0]:plan["event_windows"]["held_reused_diagnostic"][1]].copy()
assert len(train_event)>=plan["event_windows"]["minimum_train_events"]
assert len(held_event)>=plan["event_windows"]["minimum_held_events"]
assert train_event.yoy_pct_same_vintage.notna().all() and held_event.yoy_pct_same_vintage.notna().all()
assert train_event.index.max()<held_event.index.min()
# All source identity, vintage and event-calendar checks finish before reading SOXL/SOXX prices.
raw=D.features(plan["provider"]["instruments"],plan["provider"]["fields"],
    start_time=cal[0],end_time=cal[-1],freq="day").reset_index()
raw["datetime"]=pd.to_datetime(raw.datetime).dt.normalize()
assert set(raw.instrument)==set(plan["provider"]["instruments"])
assert len(raw)==len(cal)*2 and not raw.duplicated(["datetime","instrument"]).any()
opening=raw.pivot(index="datetime",columns="instrument",values="$open").reindex(cal).astype(float)
close=raw.pivot(index="datetime",columns="instrument",values="$close").reindex(cal).astype(float)
assert list(opening.columns)==["SOXL","SOXX"] and list(close.columns)==["SOXL","SOXX"]
assert opening.notna().all().all() and close.notna().all().all()
assert opening.gt(0).all().all() and close.gt(0).all().all()
soxx=close["SOXX"]
frame=pd.DataFrame(index=cal)
frame["soxx_r1_lag"]=soxx.pct_change(fill_method=None).shift(1)
frame["soxx_r5_lag"]=soxx.shift(1)/soxx.shift(6)-1
frame["soxx_rv22_lag"]=soxx.pct_change(fill_method=None).shift(1).rolling(22,min_periods=22).std(ddof=0)
frame["lever_relative_r22_lag"]=np.log(close["SOXL"].shift(1)/close["SOXL"].shift(23))-np.log(soxx.shift(1)/soxx.shift(23))
frame["ppi_yoy_same_vintage"]=events.yoy_pct_same_vintage.reindex(cal)
frame["relative_h21_label"]=np.log(opening["SOXL"].shift(-22)/opening["SOXL"].shift(-1))-np.log(opening["SOXX"].shift(-22)/opening["SOXX"].shift(-1))
frame["mature_at"]=pd.Series(cal,index=cal).shift(-22)+pd.Timedelta("9h31min")
baseline=["soxx_r1_lag","soxx_r5_lag","soxx_rv22_lag","lever_relative_r22_lag"]
candidate=baseline+["ppi_yoy_same_vintage"]
assert len(plan["feature_formula"]["baseline"])==len(baseline)
train=frame.loc[train_event.index].copy()
held=frame.loc[held_event.index].copy()
assert train[candidate+["relative_h21_label","mature_at"]].notna().all().all()
assert held[candidate+["relative_h21_label","mature_at"]].notna().all().all()
assert train.mature_at.max()<held.index.min()
assert held.mature_at.max()<=cal[-1]+pd.Timedelta("9h31min")
native=frame.loc[train.index.union(held.index),candidate+["relative_h21_label"]].copy()
native["instrument"]="SOXX";native.index.name="datetime"
native=native.reset_index().set_index(["datetime","instrument"])
panel=pd.concat({"feature":native[candidate],"label":native[["relative_h21_label"]]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(panel),segments={"all":(None,None)})
dataset.config(dump_all=True,recursive=True)
prepared=dataset.prepare("all",col_set=["feature","label"])
pd.testing.assert_frame_equal(prepared,panel)
train_index=pd.MultiIndex.from_product([train.index,["SOXX"]],names=["datetime","instrument"])
held_index=pd.MultiIndex.from_product([held.index,["SOXX"]],names=["datetime","instrument"])
y_train=prepared["label"].loc[train_index,"relative_h21_label"].to_numpy(float)
y_held=prepared["label"].loc[held_index,"relative_h21_label"].to_numpy(float)
assert np.isfinite(y_train).all() and np.isfinite(y_held).all()
(card/"run_state.json").write_text(json.dumps({"status":"preflight_complete","plan_sha256":sha(plan_path),
    "source_loads":1,"D_calendar_calls":1,"D_features_calls":1,"DatasetH_builds":1})+"\n")
tracking=card/"qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    recorder=R.get_recorder()
    (card/"run_state.json").write_text(json.dumps({"status":"fits_started","plan_sha256":sha(plan_path),
        "source_loads":1,"D_calendar_calls":1,"D_features_calls":1,"DatasetH_builds":1})+"\n")
    models={};predictions={}
    for name,features in (("baseline",baseline),("candidate",candidate)):
        scaler=StandardScaler()
        x_train=scaler.fit_transform(prepared["feature"].loc[train_index,features].to_numpy(float))
        x_held=scaler.transform(prepared["feature"].loc[held_index,features].to_numpy(float))
        assert np.isfinite(x_train).all() and np.isfinite(x_held).all() and (scaler.var_>0).all()
        model=Ridge(alpha=1.0)
        model.fit(x_train,y_train)
        models[name]={"scaler":scaler,"model":model,"features":features}
        predictions[name]=model.predict(x_held)
    held_result=held[["relative_h21_label","ppi_yoy_same_vintage","mature_at"]].copy()
    held_result["baseline_prediction"]=predictions["baseline"]
    held_result["candidate_prediction"]=predictions["candidate"]
    held_result["squared_error_improvement"]=(y_held-predictions["baseline"])**2-(y_held-predictions["candidate"])**2
    assert np.isfinite(held_result.squared_error_improvement).all()
    gate=plan["information_gate"];boot=gate["stationary_bootstrap"]
    lower=stationary_bootstrap_lower(held_result.squared_error_improvement.to_numpy(),
        mean_block=boot["mean_block_events"],draws=boot["draws"],seed=boot["seed"],quantile=boot["lower_quantile"])
    halves=[]
    for start,end in plan["event_windows"]["held_halves"]:
        vals=held_result.loc[start:end,"squared_error_improvement"]
        assert len(vals)>0
        halves.append(float(vals.mean()))
    coefficient=float(models["candidate"]["model"].coef_[-1])
    passed=bool(coefficient>0 and held_result.squared_error_improvement.mean()>0
        and all(x>0 for x in halves) and lower>0)
    result={"status":"not_promoted_source_intraday_gap" if passed else "not_promoted_information_gate_failed",
        "train_events":len(train),"held_events":len(held),"candidate_PPI_coefficient":coefficient,
        "held_mse_baseline":float(np.mean((y_held-predictions["baseline"])**2)),
        "held_mse_candidate":float(np.mean((y_held-predictions["candidate"])**2)),
        "held_paired_mse_improvement":float(held_result.squared_error_improvement.mean()),
        "held_half_improvements":halves,"stationary_bootstrap_lower":lower,"information_gate_pass":passed,
        "historical_intraday_first_receipt_qualified":False,"historical_market_independence":False,
        "new_model_fits":2,"new_bootstraps":1,"new_native_accounts":0,"qualified_forward_days":0,
        "SOXX_goal_achieved":False,"trading_or_promotion_permission":False,
        "plan_sha256":sha(plan_path),"source_recorder_id":source["recorder_id"]}
    recorder.save_objects(artifact_path="inputs",plan=plan,source_audit=source_audit)
    recorder.save_objects(artifact_path="sources",release_events=release)
    recorder.save_objects(artifact_path="models",baseline=models["baseline"],candidate=models["candidate"])
    recorder.save_objects(artifact_path="predictions",held=held_result,train_features=train[candidate+["relative_h21_label"]])
    recorder.save_objects(artifact_path="analysis",result=result)
    recorder.log_metrics(train_events=len(train),held_events=len(held),
        held_mse_improvement=result["held_paired_mse_improvement"],bootstrap_lower=lower,
        information_gate_pass=int(passed),model_fits=2,native_accounts=0)
    result.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id),recorder_uri=str(tracking),
        recorder_local_dir=str(tracking/str(recorder.experiment_id)/recorder.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(path.relative_to(card)),
    "sha256":sha(path),"bytes":path.stat().st_size} for path in sorted(artifacts.rglob("*")) if path.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(verification,indent=2)+"\n")
(card/"result_review.json").write_text(json.dumps(result,indent=2)+"\n")
summary={"experiment":plan["experiment"],"status":result["status"],"decision":result["status"],
    "result":"result_review.json","recorder_id":result["recorder_id"],"experiment_id":result["experiment_id"],
    "recorder_uri":result["recorder_uri"],"recorder_local_dir":result["recorder_local_dir"],
    "model_fits":2,"native_accounts":0,"qualified_forward_days":0,"objective_achieved":False,
    "artifact_policy":plan["artifact_policy"]}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+"\n")
(card/"conclusion.md").write_text("# 半导体 PPI 对杠杆相对收益的信息结论\n\n固定源为 93 个 ALFRED 日期版本中同版本同比，2019–2022 训练 "+str(len(train))+" 个发布事件，2023–2026 已复用诊断 "+str(len(held))+" 个事件。两次 Ridge 拟合、一次事件区块 bootstrap、零账户。候选系数 "+f"{coefficient:.6f}"+"，配对均方误差改善 "+f"{result['held_paired_mse_improvement']:.8f}"+"，下界 "+f"{lower:.8f}"+"；两个预登记持有阶段分别 "+f"{halves[0]:.8f}"+" 和 "+f"{halves[1]:.8f}"+"。信息门 "+("通过" if passed else "失败")+"。Recorder `"+result["recorder_id"]+"`。\n\n这只是 SOXL 相对 SOXX 的价格信息标签，不是费用后独立账户。ALFRED 历史盘中首收未知，2023–2026 市场历史已复用；无论得分如何，不能宣称稳定跑赢 SOXX。固定失败版本不改符号、窗口或模型。\n")
(card/"run_state.json").write_text(json.dumps({"status":"completed","plan_sha256":sha(plan_path),
    "recorder_id":result["recorder_id"]})+"\n")
print(json.dumps({"status":result["status"],"train_events":len(train),"held_events":len(held),
    "paired_improvement":result["held_paired_mse_improvement"],"lower":lower,
    "coefficient":coefficient,"recorder_id":result["recorder_id"]}))


## 中文结论

执行后以同卡 `conclusion.md`、`result_review.json`、`summary.json` 和 Qlib Recorder 为准。执行前没有因子、账户或晋升结果。
